# 강의 04 · 실습 1 — 상태·노드·조건부 엣지 · (6) 고난도 III

## 1. 문제상황

- 사내 IT 헬프데스크의 일반 요청은 담당 부서가 셋으로 나뉩니다. 로그인·비밀번호·권한은 계정 담당, PC·모니터·프린터 고장은 장비 담당, 와이파이·VPN·유선 연결은 네트워크 담당입니다.
- 부서마다 안내문에 담아야 할 내용과 말투가 달라서, 담당자는 요청을 읽고 부서를 먼저 정한 뒤 그 부서의 안내문 양식대로 글을 씁니다. 부서를 잘못 고르면 요청자는 엉뚱한 안내를 받고 다시 문의합니다.
- 긴급 요청은 엔지니어를 호출하고 접수 기록에 기록한 뒤, 관리자에게 따로 통지해야 합니다. 일반 요청은 기록으로 끝납니다.
- 담당자는 요청마다 긴급도 판단, 부서 판단, 부서별 안내문 작성, 기록, 관리자 통지 여부 판단을 반복합니다. 판단이 셋으로 늘어나면서 빠뜨리는 단계가 생깁니다.

## 2. 문제와 목표

- **문제**: 요청 한 건에 판단이 셋(긴급도, 담당 부서, 관리자 통지 여부) 들어가고, 판단마다 분기가 다릅니다. 사람이 반복하면 부서를 잘못 고르거나 통지를 빠뜨립니다.
- **목표**
  - 요청 한 건을 입력하면 프로그램이 긴급도를 판정하고, 긴급이면 엔지니어 호출 메시지를 만듭니다.
    - 긴급 판정의 기준: 서비스가 멈추었거나 여러 사람이 일을 못 하면 긴급, 그 밖은 일반
  - 일반이면 담당 부서를 판정한 뒤 부서별 안내 노드 셋 중 하나가 안내문을 만듭니다.
    - 담당 부서 셋: 계정(로그인·비밀번호·권한), 장비(PC·모니터·프린터 고장), 네트워크(와이파이·VPN·유선 연결)
  - 모든 요청이 접수 기록을 거친 뒤, 긴급 요청만 관리자 통지 노드를 거쳐 끝나는 처리 흐름을 만듭니다.
    - 상태에 담을 정보 여섯: 요청 본문, 긴급도 판정 결과, 담당 부서, 요청자에게 나갈 글, 기록 여부, 관리자 통지 여부 (키의 이름과 타입은 학생이 정함)
    - 노드 구성: 긴급도 판정, 엔지니어 호출 메시지 작성, 담당 부서 판정, 부서별 안내문 작성 셋, 접수 기록, 관리자 통지 (노드 이름은 학생이 정함, 부서별 안내 노드 셋은 상태의 같은 키에 쓰며, 기록과 통지는 화면 출력으로 대신함)
    - 접수 요청 세 건의 문면은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**:
  - 긴급 요청 한 건, 계정 관련 일반 요청 한 건, 네트워크 관련 일반 요청 한 건을 입력했을 때,
  - 세 요청이 거친 노드 열이 서로 다르고, 긴급 요청만 호출 메시지 노드와 관리자 통지 노드를 거치며,
  - 일반 요청 둘은 각각 자기 부서의 안내 노드를 거친 뒤 기록 노드에서 끝나고, 최종 상태의 관리자 통지 여부가 긴급 요청만 참인 것을 실행 결과에서 확인합니다.
    - 요청마다 거친 노드의 열과 최종 상태(긴급도·부서·기록 여부·통지 여부)를 한 줄로 출력합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

이 단에서는 요구사항을 주지 않습니다. 「1. 문제상황」「2. 문제와 목표」와 「3. 워크플로우 다이어그램」에 직접 그린 다이어그램을 보고 요구사항을 번호 목록으로 적습니다. 노드마다 「무엇을 읽고 어느 키에 쓰는가」를 한 항목으로, 조건부 엣지마다 「어느 키를 보고 어느 이름을 돌려주는가」를 한 항목으로 적습니다.

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")


# 주어진 자료
TICKETS = [
    "사내 그룹웨어가 오전 9시부터 접속되지 않습니다. 부서 전체가 결재를 올리지 못하고 있습니다.",
    "계정이 잠겨서 로그인이 되지 않습니다. 비밀번호를 여러 번 틀린 것 같습니다.",
    "사무실 유선 인터넷이 제 자리만 되지 않습니다. 옆자리는 됩니다.",
]


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

실행 결과에서 다음 세 가지를 확인합니다.

1. 세 접수가 거친 노드 열이 서로 다릅니다. 1번 접수(긴급)는 호출 메시지 노드와 통지 노드를 거치고, 2번 접수는 계정 안내 노드를, 3번 접수는 네트워크 안내 노드를 거칩니다.
2. 최종 상태의 관리자 통지 여부가 1번 접수만 참이고 나머지는 `None`입니다. 기록 노드 뒤의 조건부 엣지가 긴급 건만 통지 노드로 보냈다는 뜻입니다.
3. 2번과 3번 접수의 부서 판정 결과가 각각 계정과 네트워크이고, 거친 안내 노드가 그 부서와 짝지어져 있습니다.